# BuffettRAG: LoRA fine-tune Qwen2.5-7B-Instruct (FT-r3, ft_v5) + one GGUF per epoch
One adapter per epoch (valid loss evaluated each epoch), q4_k_m GGUF exported for epoch 1 and epoch 2. Re-run top to bottom after a dropped session: training resumes from the last checkpoint, finished exports are skipped.

## a. Settings

In [ ]:
DATA_NAME = 'ft_v5'  # 'ft_v5' (default) or 'ft_v4'
assert DATA_NAME in ('ft_v4', 'ft_v5'), DATA_NAME
DRIVE_DIR = '/content/drive/MyDrive/buffettrag'
BUNDLE = DRIVE_DIR + '/colab-bundle-7b.zip'
OUT_DIR = DRIVE_DIR + '/run-7b-r3'
BASE_MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
MAX_SEQ = 6144  # settings variable; the Kaggle build script may override it
EPOCHS = 2
LR = 1e-4
RANK = 16
ALPHA = 32
BATCH = 1
GRAD_ACCUM = 2
SEED = 13
EXPORT_BASE_GGUF = False

## b. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > GPU, then re-run.')
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)

## c. Mount Drive, unzip bundle, verify SHA256SUMS

In [ ]:
import hashlib, os, zipfile
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isfile(BUNDLE), 'bundle not found: ' + BUNDLE
with zipfile.ZipFile(BUNDLE) as zf:
    zf.extractall('/content/bundle')
bad = 0
for line in open('/content/bundle/SHA256SUMS'):
    if not line.strip():
        continue
    want, rel = line.strip().split(None, 1)
    got = hashlib.sha256(open('/content/bundle/' + rel.strip(), 'rb').read()).hexdigest()
    ok = got == want
    bad += not ok
    print('OK ' if ok else 'BAD', rel)
if bad:
    raise SystemExit(f'{bad} checksum mismatch(es); stopping.')

## d. Install unsloth

In [ ]:
!pip install -q unsloth
import torch, transformers, trl, unsloth
print('torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__, '| unsloth', getattr(unsloth, '__version__', '?'))

## e. Load base model + LoRA

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=RANK, lora_alpha=ALPHA, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)

## f. Data (assistant-only loss)

In [ ]:
import json
from datasets import Dataset

def load(path):
    return [json.loads(l)['messages'] for l in open(path) if l.strip()]

def prep(rows, name):
    keep, dropped, longest = [], 0, 0
    for msgs in rows:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        n = len(tokenizer(text, add_special_tokens=False)['input_ids'])
        longest = max(longest, n)
        if n > MAX_SEQ:
            dropped += 1
        else:
            keep.append({'text': text})
    print(f'{name}: rows={len(rows)} kept={len(keep)} dropped(>{MAX_SEQ})={dropped} max_tokens={longest}')
    return Dataset.from_list(keep), dropped

train_ds, TRAIN_DROPPED = prep(load(f'/content/bundle/{DATA_NAME}/train.jsonl'), 'train')
valid_ds, VALID_DROPPED = prep(load(f'/content/bundle/{DATA_NAME}/valid.jsonl'), 'valid')

## g. Train (resumes from checkpoint if present)

In [ ]:
import glob, time
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.chat_templates import train_on_responses_only

CKPT_DIR = OUT_DIR + '/checkpoints'
args = SFTConfig(
    output_dir=CKPT_DIR, dataset_text_field='text', max_seq_length=MAX_SEQ, packing=False,
    num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, eval_strategy='epoch',
    per_device_eval_batch_size=1, logging_steps=5, save_strategy='epoch',
    save_total_limit=max(2, EPOCHS), fp16=not is_bf16_supported(), bf16=is_bf16_supported(), seed=SEED,
    optim='adamw_8bit', lr_scheduler_type='linear', warmup_steps=5, weight_decay=0.01, report_to='none')
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=valid_ds, args=args)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
resume = bool(glob.glob(CKPT_DIR + '/checkpoint-*'))
print('resume from checkpoint:', resume)
t0 = time.time()
result = trainer.train(resume_from_checkpoint=True) if resume else trainer.train()
SECONDS = round(time.time() - t0, 1)
print(result)

## h. Save final adapter, one adapter per epoch, metrics

In [ ]:
import shutil
model.save_pretrained(OUT_DIR + '/adapter')
tokenizer.save_pretrained(OUT_DIR + '/adapter')

# One adapter per epoch: the epoch-end checkpoints are kept (never deleted); copy their adapter files next to the final adapter.
EPOCH_ADAPTERS = {}
for d in sorted(glob.glob(CKPT_DIR + '/checkpoint-*')):
    ep = round(json.load(open(d + '/trainer_state.json'))['epoch'])
    assert os.path.isfile(d + '/adapter_model.safetensors'), 'no adapter in ' + d
    dst = OUT_DIR + f'/adapter_e{ep}'
    shutil.copytree(d, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns('optimizer.pt', 'scheduler.pt', 'rng_state*', 'scaler.pt'))
    EPOCH_ADAPTERS[ep] = dst
assert sorted(EPOCH_ADAPTERS) == list(range(1, EPOCHS + 1)), f'epoch adapters found: {sorted(EPOCH_ADAPTERS)}'
print('epoch adapters:', EPOCH_ADAPTERS)

hist = trainer.state.log_history
train_losses = [h['loss'] for h in hist if 'loss' in h]
metrics = {
    'gpu': GPU_NAME, 'torch': torch.__version__, 'unsloth': getattr(unsloth, '__version__', '?'),
    'data_name': DATA_NAME, 'max_seq': MAX_SEQ,
    'epochs': EPOCHS, 'steps': trainer.state.global_step, 'train_rows': len(train_ds),
    'dropped_rows': {'train': TRAIN_DROPPED, 'valid': VALID_DROPPED},
    'final_train_loss': result.training_loss if result.training_loss is not None else (train_losses[-1] if train_losses else None),
    'last_logged_train_loss': train_losses[-1] if train_losses else None,
    'eval_loss_per_epoch': [{'epoch': round(h['epoch']), 'step': h['step'], 'eval_loss': h['eval_loss']} for h in hist if 'eval_loss' in h],
    'eval_loss_history': [{'step': h['step'], 'eval_loss': h['eval_loss']} for h in hist if 'eval_loss' in h],
    'seconds': SECONDS, 'seconds_note': 'this session only (a resumed run counts the resumed part)',
}
json.dump(metrics, open(OUT_DIR + '/metrics.json', 'w'), indent=2)
print(json.dumps(metrics, indent=2))

## i. Export one merged GGUF (q4_k_m) per epoch adapter

In [ ]:
import glob, shutil, gc
EPOCH_GGUFS = {
    1: OUT_DIR + '/buffett-qwen2.5-7b-ft-r3-e1-q4_k_m.gguf',
    2: OUT_DIR + '/buffett-qwen2.5-7b-ft-r3-e2-q4_k_m.gguf',
}
assert EPOCHS == 2, 'two epoch GGUFs (e1, e2) are exported; EPOCHS must be 2'

_sha_cache = {}
def sha256_file(p):
    key = (p, os.path.getsize(p))
    if key not in _sha_cache:
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 24), b''):
                h.update(b)
        _sha_cache[key] = h.hexdigest()
    return _sha_cache[key]

def write_sums():
    with open(OUT_DIR + '/SHA256SUMS', 'w') as f:
        for p in sorted(glob.glob(OUT_DIR + '/*.gguf')):
            f.write(f'{sha256_file(p)}  {os.path.basename(p)}\n')

def export_gguf(m, tok, work, dest):
    # unsloth merges the LoRA into a temp dir relative to the cwd, so work in /tmp and clean up after copying
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(work)
    cwd = os.getcwd()
    os.chdir(work)
    try:
        m.save_pretrained_gguf(work + '/out', tok, quantization_method='q4_k_m')
        cands = [p for p in glob.glob(work + '*/**/*.gguf', recursive=True) if 'q4_k_m' in os.path.basename(p).lower()]
        assert len(cands) == 1, f'expected one q4_k_m gguf, found {cands}'
        shutil.copy(cands[0], dest)
    finally:
        os.chdir(cwd)
        shutil.rmtree(work, ignore_errors=True)
        for p in glob.glob(work + '*'):
            shutil.rmtree(p, ignore_errors=True)

# the training model is no longer needed; each epoch adapter is reloaded on its own
globals().pop('trainer', None); globals().pop('model', None)
gc.collect(); torch.cuda.empty_cache()

for n in sorted(EPOCH_GGUFS):
    dest = EPOCH_GGUFS[n]
    if os.path.exists(dest):
        print('exists, skipping:', dest)
    else:
        m, tok = FastLanguageModel.from_pretrained(EPOCH_ADAPTERS[n], max_seq_length=MAX_SEQ, load_in_4bit=True)
        try:
            export_gguf(m, tok, f'/tmp/gguf_ft_e{n}', dest)
        finally:
            del m, tok
            gc.collect(); torch.cuda.empty_cache()
    print(dest, os.path.getsize(dest), 'bytes\nsha256', sha256_file(dest))
write_sums()

## j. Export untouched base GGUF (same tool chain, off by default)

In [ ]:
BASE_GGUF = OUT_DIR + '/qwen2.5-7b-instruct-q4_k_m.gguf'
if not EXPORT_BASE_GGUF:
    print('EXPORT_BASE_GGUF is False, skipping')
elif os.path.exists(BASE_GGUF):
    print('exists, skipping:', BASE_GGUF)
else:
    base_model, base_tok = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True)
    export_gguf(base_model, base_tok, '/tmp/gguf_base', BASE_GGUF)
    del base_model, base_tok
    gc.collect(); torch.cuda.empty_cache()
if os.path.exists(BASE_GGUF):
    print(BASE_GGUF, os.path.getsize(BASE_GGUF), 'bytes\nsha256', sha256_file(BASE_GGUF))
write_sums()
print(open(OUT_DIR + '/SHA256SUMS').read())

## k. Done

In [ ]:
print('ALL_DONE')
for root, _, files in os.walk(OUT_DIR):
    if '/checkpoints' in root:
        continue
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p):>14,}  {p}')